In [1]:
import base64
import json
import os
import re
import sys
from pathlib import Path

import cv2
import whisper
from dotenv import find_dotenv, load_dotenv
from moviepy import VideoFileClip
from openai import OpenAI


In [2]:
load_dotenv(find_dotenv(), override=True)

PROJECT_DIR = Path.cwd()
print(PROJECT_DIR)

s:\Project\video_structure_transform\backend\my-video


In [3]:
VIDEO_FILENAME = "抖音2026529-207530.mp4"
VIDEO_PATH = PROJECT_DIR.parent / "tests" / "videos" / VIDEO_FILENAME

In [4]:
CACHE_DIR = PROJECT_DIR / ".pipeline_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR = PROJECT_DIR / "audio_file"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_AUDIO_PATH = AUDIO_DIR / "temp_audio.wav"

# 输出文件路径（全部在 my-video/ 下）
DESIGN_MD_PATH = PROJECT_DIR / "DESIGN.md"
SCRIPT_MD_PATH = PROJECT_DIR / "SCRIPT.md"
STORYBOARD_MD_PATH = PROJECT_DIR / "STORYBOARD.md"
NARRATION_TXT_PATH = PROJECT_DIR / "narration.txt"
TRANSCRIPT_JSON_PATH = PROJECT_DIR / "transcript.json"
COMPOSITIONS_DIR = PROJECT_DIR / "compositions"
INDEX_HTML_PATH = PROJECT_DIR / "index.html"

COMPOSITIONS_DIR.mkdir(parents=True, exist_ok=True)

client = OpenAI(
    api_key=os.getenv("API_KEY", ""),
    base_url=os.getenv("BASE_URL", ""),
)
MODEL = os.getenv("MODEL", "seed-2.0-lite")


In [5]:
def extract_audio_and_transcript(video_path: str, audio_out: str):
    print("📢 正在提取音频...")
    video = VideoFileClip(video_path)
    video.audio.write_audiofile(audio_out, logger=None)
    video.close()

    print("🗣️  正在 ASR 转录...")
    whisper_model = whisper.load_model("base")
    result = whisper_model.transcribe(audio_out, word_timestamps=True)

    segments = [
        {
            "start": round(seg["start"], 2),
            "end": round(seg["end"], 2),
            "text": seg["text"],
        }
        for seg in result["segments"]
    ]
    words = []
    for seg in result["segments"]:
        for w in seg.get("words", []):
            words.append(
                {
                    "text": w["word"].strip(),
                    "start": round(w["start"], 3),
                    "end": round(w["end"], 3),
                }
            )
    return segments, result["text"], words


In [6]:
def analyze_video_rhythm(video_path: str, threshold: float = 30.0):
    print("🎬 正在分析视频节奏...")
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    ret, prev_frame = cap.read()
    if not ret:
        cap.release()
        return [], fps, 0.0

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    cut_timestamps = []
    frame_index = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_index += 1
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if cv2.absdiff(gray, prev_gray).mean() > threshold:
            cut_timestamps.append(round(frame_index / fps, 2))
        prev_gray = gray

    cap.release()
    return cut_timestamps, fps, frame_index / fps


In [7]:
def encode_video_base64(video_path: str) -> str:
    print("💾 正在将视频编码为 base64...")
    with open(video_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


In [8]:
def _call_multimodal(
    system_prompt: str,
    user_text: str,
    video_b64: str,
    max_tokens: int = 4096,
    temperature: float = 0.3,
) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {
                        "type": "video_url",
                        "video_url": {"url": f"data:video/mp4;base64,{video_b64}"},
                    },
                    {"type": "text", "text": user_text},
                ],
            },
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return completion.choices[0].message.content.strip()


In [9]:
def _call_text_only(
    system_prompt: str, user_text: str, max_tokens: int = 4096, temperature: float = 0.3
) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return completion.choices[0].message.content.strip()


In [10]:
DESIGN_SYSTEM_PROMPT = (
    "你是一位顶尖的品牌视觉设计师和 Hyperframes 视频生产专家。"
    "根据所提供的爆款短视频，输出一份符合 Hyperframes DESIGN.md 规范的品牌设计文档。"
    "请直接输出 Markdown 文本，不要添加任何额外解释或代码块标记。"
)


def build_design_prompt(segments, total_text, cuts, duration):
    segs = "\n".join(f"[{s['start']}s-{s['end']}s]: {s['text']}" for s in segments)
    cuts_str = ", ".join(f"{t}s" for t in cuts[:30])
    return f"""请仔细观看这段爆款短视频，结合以下数据，生成一份完整的 DESIGN.md 文件。

【视频基础数据】
- 时长：{duration:.1f}s | 切换点：{cuts_str}

【台词】
{total_text}

【台词时间戳】
{segs}

【输出格式】严格按此结构输出，包含 YAML frontmatter + 6个章节：

---
colors:
  primary: "#xxxxxx"
  on-primary: "#xxxxxx"
  accent: "#xxxxxx"
  surface: "#xxxxxx"
  muted: "#xxxxxx"
typography:
  headline:
    fontFamily: "Noto Serif SC"
    fontSize: "5rem"
    fontWeight: 700
    textTransform: "none"
  body:
    fontFamily: "Noto Sans SC"
    fontSize: "1.5rem"
    fontWeight: 400
    lineHeight: 1.6
  label:
    fontFamily: "Noto Sans SC"
    fontSize: "1rem"
    fontWeight: 500
rounded:
  sm: "8px"
  md: "16px"
  lg: "32px"
spacing:
  sm: "16px"
  md: "32px"
  lg: "64px"
motion:
  energy: "high"
  easing:
    entry: "power3.out"
    exit: "power2.in"
    ambient: "sine.inOut"
  duration:
    entrance: 0.4
    hold: 2.0
    transition: 0.5
  atmosphere:
    - "radial-glow"
    - "ghost-type"
  transition: "velocity-matched-upward"
---

## Overview
（3-4句话）

## Colors
（5-8个颜色含十六进制）

## Typography
（字体层级说明，字体名只能使用 Google Fonts 中存在的英文名称，如 Noto Serif SC、Noto Sans SC、Inter 等，不要使用中文字体名）

## Components
（视频中的视觉组件）

## Imagery
（画面素材类型）

## Do's and Don'ts
**Do:**
- （3-5条）
**Don't:**
- （3-5条）

重要：字体名必须是 Google Fonts 可用的英文名称，严禁使用"思源黑体"、"思源宋体"等中文名。"""


In [11]:
SCRIPT_SYSTEM_PROMPT = (
    "你是一位精通短视频创作的脚本策划专家和 Hyperframes 视频生产专家。"
    "根据所提供的爆款短视频，提炼脚本结构，输出符合 Hyperframes SCRIPT.md 规范的脚本文档。"
    "请直接输出 Markdown 文本，不要添加任何额外解释或代码块标记。"
)


def build_script_prompt(segments, total_text, cuts, duration, design_content):
    segs = "\n".join(f"[{s['start']}s-{s['end']}s]: {s['text']}" for s in segments)
    cuts_str = ", ".join(f"{t}s" for t in cuts[:30])
    return f"""请仔细观看这段爆款短视频，结合以下数据，生成一份完整的 SCRIPT.md 文件。

【DESIGN.md（参考）】
{design_content[:600]}...

【视频数据】时长 {duration:.1f}s | 切换点：{cuts_str}

【完整台词】
{total_text}

【台词时间戳】
{segs}

【输出格式】

# SCRIPT.md

## Global Direction
- **Video type**: （产品带货 / 情感共鸣 / 知识科普 / 生活方式）
- **Target audience**: （目标受众）
- **Platform**: （抖音 / 小红书 / 视频号）
- **Tone**: （语气风格）
- **Total duration**: {duration:.1f}s

## Narration

### Hook（0.0s - Xs）
> （原台词还原）
**Template**: "{{{{主题}}}} {{{{痛点/卖点}}}}……"
**Formula**: （hook 结构分析）

---

### Story（Xs - Xs）
> （原台词还原）
**Template**: "（带占位符的句式）"
**Beat breakdown**:
- Beat 1 (Xs-Xs): （做了什么）

---

### Proof（Xs - Xs）
> （原台词还原）
**Template**: "（带占位符的句式）"
**Proof type**: （证明类型）

---

### CTA（Xs - {duration:.1f}s）
> （原台词还原）
**Template**: "（带占位符的句式）"
**CTA type**: （关注 / 购买 / 评论）

---

## Rhythm Notes
- **Pacing**: （快/中/慢）
- **Emotional beats**: （情绪高点时间节点）
- **Music cues**: （BGM 卡点时间）

## Migration Instructions
（迁移说明）"""


In [12]:
STORYBOARD_SYSTEM_PROMPT = (
    "你是一位精通 Hyperframes 视频制作的分镜导演和运动设计师。"
    "根据所提供的爆款短视频及已生成的 DESIGN.md / SCRIPT.md，"
    "输出一份完整、可直接执行的 STORYBOARD.md 文件。"
    "请直接输出 Markdown 文本，不要添加任何额外解释或代码块标记。"
)


def build_storyboard_prompt(segments, cuts, duration, design_content, script_content):
    segs = "\n".join(f"[{s['start']}s-{s['end']}s]: {s['text']}" for s in segments)
    all_times = sorted(set([0.0] + cuts[:20] + [duration]))
    beats_preview = " → ".join(f"{t}s" for t in all_times[:15])
    return f"""请仔细观看这段爆款短视频，结合以下数据，生成完整的 STORYBOARD.md 文件。

【DESIGN.md】
{design_content}

【SCRIPT.md（摘要）】
{script_content[:1000]}...

【时间线】总时长 {duration:.1f}s | Beat边界: {beats_preview}
【台词时间戳】
{segs}

【输出格式】

# STORYBOARD.md

## Global Direction
- **Format**: 9:16 vertical, {int(duration)}s
- **Visual style**: （引用 DESIGN.md 风格）
- **Scene rhythm**: （命名节奏，如 hook-PUNCH-breathe-CTA）
- **Primary transition**: velocity-matched-upward
- **Guardrails**: （全局约束）

---

## Beat-by-Beat Direction

### Beat 1 — [Hook] (0.0s - Xs)

**Composition filename**: `beat-1-hook.html`
**Narration**: "（台词）"
**Concept**: （2-3句）
**Mood & camera**: （镜头感）

**Depth layers**:
- BG: （颜色 + 2-5个装饰元素）
- MG: （主要内容）
- FG: （字幕条、贴纸等）

**Animation choreography**:
- `主标题`: SLAMS in from left, 0.3s, expo.out
- `字幕条`: FADES up from bottom, 0.4s, power2.out
- `背景光晕`: breathe scale 1.0→1.08, 4s loop, sine.inOut

**Techniques**: （选2-3个：SVG Path Drawing / Canvas 2D / CSS 3D / Per-Word Typography / Character Typing / Velocity Transitions）

**Transition out**: velocity-matched upward — exit y:-150 blur:30px 0.33s power2.in

**SFX**: 不使用外部音效文件（项目内无 /assets/sfx/ 目录）

---

### Beat 2 — [Story] (Xs - Xs)
**Composition filename**: `beat-2-story.html`
（同上格式）

---

### Beat N — [CTA] (Xs - {duration:.1f}s)
**Composition filename**: `beat-N-cta.html`
（同上格式，最后一个 beat 允许 fade-out）

---

## Asset Audit

| Asset | Type | Used in | Notes |
|-------|------|---------|-------|
| （文件名） | image/video/font | Beat X | （用途） |

## Migration Notes
（迁移说明）

重要约束：
- 字体只使用 Google Fonts 英文名（Noto Serif SC、Noto Sans SC、Inter 等），不用中文名
- SFX 字段一律填"不使用外部音效文件"，因为项目无 /assets/sfx/ 目录"""


In [13]:
NARRATION_CLEAN_SYSTEM_PROMPT = (
    "你是一位专业的视频配音脚本编辑。"
    "将输入的原始 ASR 台词整理为干净、流畅的旁白文本，并做发音替换：\n"
    "- 英文缩写拆字母（API → A P I，URL → U R L）\n"
    "- 数字/金额用汉字表达（$2T → 两万亿，100% → 百分之百）\n"
    "- 去除口语填充词（嗯、啊、那个等）\n"
    "- 保持原始语气和节奏，不改变句义。\n"
    "只输出处理后的纯文本，不加任何说明。"
)


def generate_narration_txt(total_text: str) -> str:
    return _call_text_only(NARRATION_CLEAN_SYSTEM_PROMPT, total_text, max_tokens=2048)


def build_transcript_json(words: list) -> list:
    return words


In [14]:
def _parse_beats_from_storyboard(storyboard_content: str) -> list:
    beats = []
    pattern = re.compile(
        r"###\s+Beat\s+(\d+)\s*[—–-]\s*\[([^\]]+)\]\s*\((\d+\.?\d*)s\s*-\s*(\d+\.?\d*)s\)",
        re.IGNORECASE,
    )
    fn_pattern = re.compile(r"\*\*Composition filename\*\*:\s*`([^`]+)`")
    narration_pattern = re.compile(r"\*\*Narration\*\*:\s*[\"「]([^\"「」\n]+)[\"」]?")

    lines = storyboard_content.split("\n")
    current_beat = None

    for line in lines:
        m = pattern.search(line)
        if m:
            if current_beat:
                beats.append(current_beat)
            num, label, start, end = m.groups()
            safe_label = re.sub(r"[^a-zA-Z0-9]", "-", label.lower())
            beat_id = f"beat-{num}-{safe_label}"
            current_beat = {
                "num": int(num),
                "id": beat_id,
                "label": label,
                "start": float(start),
                "end": float(end),
                "filename": f"{beat_id}.html",
                "narration": "",
            }
            continue

        if current_beat:
            fn_m = fn_pattern.search(line)
            if fn_m:
                fname = fn_m.group(1).strip()
                current_beat["filename"] = fname
                current_beat["id"] = fname.replace(".html", "")

            nar_m = narration_pattern.search(line)
            if nar_m:
                current_beat["narration"] = nar_m.group(1).strip()

    if current_beat:
        beats.append(current_beat)

    return beats


In [15]:
def _extract_design_tokens(design_content: str) -> dict:
    """提取 DESIGN.md 中的关键 token，用于代码层面校验/注入。"""
    tokens = {
        "primary": "#0a0a0a",
        "on_primary": "#ffffff",
        "accent": "#c84f1c",
        "headline_font": "Noto Serif SC",
        "body_font": "Noto Sans SC",
    }
    for line in design_content.split("\n"):
        l = line.strip()
        if l.startswith("primary:") and "#" in l:
            m = re.search(r"#[0-9a-fA-F]{6}", l)
            if m:
                tokens["primary"] = m.group()
        elif l.startswith("on-primary:") and "#" in l:
            m = re.search(r"#[0-9a-fA-F]{6}", l)
            if m:
                tokens["on_primary"] = m.group()
        elif l.startswith("accent:") and "#" in l:
            m = re.search(r"#[0-9a-fA-F]{6}", l)
            if m:
                tokens["accent"] = m.group()
        elif "fontFamily:" in l:
            m = re.search(r'fontFamily:\s*["\']?([^"\'#\n]+)["\']?', l)
            if m:
                font = m.group(1).strip().strip("\"'")
                if (
                    "headline"
                    in design_content[
                        max(0, design_content.find(l) - 200) : design_content.find(l)
                    ]
                ):
                    tokens["headline_font"] = font
                else:
                    tokens["body_font"] = font
    return tokens


In [16]:
def _postprocess_composition_html(html: str, comp_id: str, beat_duration: float) -> str:
    """
    对 LLM 生成的 HTML 做规则性修复，消灭所有可预测的 lint error：
    1. [ERROR] timeline_registry_missing_init → 确保 window.__timelines 初始化在赋值前
    2. [ERROR] media_missing_data_start → 给所有 <audio> 补 data-start data-duration data-track-index
    3. [ERROR] audio_src_not_found → 删除引用不存在 sfx 文件的 <audio> 元素
    4. [WARN]  gsap_repeat_ceil_overshoot → 将 Math.ceil 替换为 Math.floor（用于 repeat 计算）
    5. [WARN]  studio_missing_editable_id → 给所有有 data-track-index 但无 id 的元素补 id
    6. [WARN]  font_family_without_font_face → 将中文字体名替换为 Google Fonts 英文名
    7. [ERROR] root_composition_missing_data_start → 在 root comp div 上补 data-start="0"（仅 index.html）
    """

    # ── 1. 确保 window.__timelines 初始化紧靠在赋值语句之前 ──────
    # 先删除所有已存在的初始化行，再在第一个赋值前统一插入
    html = re.sub(
        r"window\.__timelines\s*=\s*window\.__timelines\s*\|\|\s*\{\};\s*\n?", "", html
    )
    html = re.sub(
        r'(window\.__timelines\[")',
        r"window.__timelines = window.__timelines || {};\n      \1",
        html,
        count=1,
    )

    # ── 2. 删除引用不存在 sfx/音效文件的 <audio> 标签 ────────────
    # 只保留：没有 src 属性、或 src 不含路径分隔符（即本地根目录文件）
    def _should_keep_audio(tag: str) -> bool:
        src_m = re.search(r'src=["\']([^"\']+)["\']', tag)
        if not src_m:
            return True  # 无 src，保留
        src = src_m.group(1)
        # 含目录路径且文件不存在 → 删除
        if "/" in src or "\\" in src:
            candidate = PROJECT_DIR / src.lstrip("./")
            return candidate.exists()
        # 根目录文件
        return (PROJECT_DIR / src).exists()

    def _fix_audio_tag(tag: str, idx: int, beat_dur: float) -> str:
        """给 <audio> 补齐必需属性。"""
        if "data-start" not in tag:
            tag = tag.replace("<audio", '<audio data-start="0"', 1)
        if "data-duration" not in tag:
            tag = tag.replace("<audio", f'<audio data-duration="{beat_dur}"', 1)
        if "data-track-index" not in tag:
            tag = tag.replace("<audio", f'<audio data-track-index="9{idx}"', 1)
        return tag

    audio_tags = re.findall(r"<audio[^>]*/?>", html, re.IGNORECASE)
    for i, atag in enumerate(audio_tags):
        if not _should_keep_audio(atag):
            html = html.replace(atag, "", 1)
        else:
            fixed = _fix_audio_tag(atag, i, beat_duration)
            if fixed != atag:
                html = html.replace(atag, fixed, 1)

    # 同时清理孤立的 </audio> 结束标签（删除 audio 后留下的）
    # （如果是 <audio>...</audio> 块）
    html = re.sub(
        r"<audio[^>]*/assets/sfx[^>]*>.*?</audio>",
        "",
        html,
        flags=re.DOTALL | re.IGNORECASE,
    )

    # ── 3. Math.ceil → Math.floor（repeat 计算防止 overshoot）────
    html = re.sub(
        r"Math\.ceil\((\w+)\s*/\s*(\w+)\)\s*-\s*1", r"Math.floor(\1 / \2) - 1", html
    )

    # ── 4. 补 id：给有 data-track-index 但无 id 的元素加 id ───────
    def _add_missing_id(m: re.Match) -> str:
        tag = m.group(0)
        if "id=" in tag:
            return tag
        ti_m = re.search(r'data-track-index=["\'](\d+)["\']', tag)
        ti = ti_m.group(1) if ti_m else "x"
        new_id = f"{comp_id}-el-{ti}"
        return tag.replace("<div", f'<div id="{new_id}"', 1)

    html = re.sub(r"<div[^>]+data-track-index[^>]*>", _add_missing_id, html)

    # ── 5. 中文字体名 → Google Fonts 英文名 ──────────────────────
    FONT_MAP = {
        "思源黑体": "Noto Sans SC",
        "思源宋体": "Noto Serif SC",
        "思源粗宋": "Noto Serif SC",
        "方正黑体": "Noto Sans SC",
        "方正宋体": "Noto Serif SC",
        "微软雅黑": "Noto Sans SC",
        "苹方": "Noto Sans SC",
        "黑体": "Noto Sans SC",
        "宋体": "Noto Serif SC",
        "old english text mt": "Cinzel",
        "Old English Text MT": "Cinzel",
    }
    for zh, en in FONT_MAP.items():
        html = html.replace(zh, en)

    return html


In [17]:
COMPOSITION_HTML_SYSTEM_PROMPT = """\
你是一位精通 Hyperframes + GSAP 的视频合成工程师。
根据提供的 beat 分镜描述、DESIGN.md 和台词时间戳，
生成一个符合 Hyperframes 规范的 sub-composition HTML 文件。

━━━━━━━━━━━━━━ 必须严格遵守的规范 ━━━━━━━━━━━━━━

【结构】
- 整个文件用 <template id="{comp_id}-template"> 包裹
- composition div 必须有：data-composition-id="{comp_id}" data-width="1080" data-height="1920"
- 每个 clip 元素必须同时有：id="唯一id" class="clip" data-start="秒" data-duration="秒" data-track-index="唯一整数"
  ★ id 是必须的，格式建议：{comp_id}-bg / {comp_id}-title / {comp_id}-subtitle 等
  ★ data-track-index 在整个 composition 内必须唯一，从 0 开始递增

【Timeline】
- 必须首先写：window.__timelines = window.__timelines || {};
- 然后：const tl = gsap.timeline({{ paused: true }});
- 最后：window.__timelines["{comp_id}"] = tl;
- 所有动画挂在 tl 上，包括 ambient loop（禁止裸 gsap.to()）
- 用 tl.fromTo() 而非 tl.from()（防止 immediateRender 问题）
- repeat 必须有限：Math.floor(duration / cycleDuration) - 1（禁止 repeat: -1）
  注意：用 Math.floor 不是 Math.ceil
- 第一个入场动画从 t=0.1 开始，不从 0 开始

【字体】
- 只使用 Google Fonts 中的字体，必须用英文名：
  ✓ Noto Serif SC（中文宋/serif）
  ✓ Noto Sans SC（中文黑/sans）
  ✓ Inter（英文 sans）
  ✓ Cinzel（英文 decorative）
  ✗ 禁止：思源黑体、思源宋体、思源粗宋、方正、微软雅黑、苹方、Old English Text MT

【音效/音频】
- 禁止在 sub-composition 里添加 <audio> 元素（BGM 由 index.html 统一管理）
- 如果 storyboard 有 SFX 描述，用 GSAP 动画效果模拟（如 scale burst），不用 <audio>

【动画规范】
- 标题 ≥ 80px，正文 ≥ 32px，标签 ≥ 24px
- 至少 3 种不同 GSAP ease
- 至少 8 个视觉元素（BG 装饰 2-5 个 + MG 内容 + FG 细节）
- 装饰元素必须有 ambient GSAP loop（呼吸/漂移/脉冲），挂在 tl 上

【禁止】
- Math.random() / Date.now()
- repeat: -1
- 异步构建 timeline（async/setTimeout/Promise）
- 在 <audio> 里引用 /assets/sfx/ 等不存在的路径
- 非最后 beat 禁止出场动画（opacity→0 / y offscreen）

只输出完整 HTML，从 <template> 开始，到 </template> 结束，不加任何 markdown 包裹或说明。"""


def _build_composition_prompt(
    beat: dict,
    design_content: str,
    storyboard_beat_section: str,
    transcript_words: list,
    is_final: bool,
) -> str:
    beat_words = [
        w
        for w in transcript_words
        if w["start"] >= beat["start"] - 0.1 and w["end"] <= beat["end"] + 0.1
    ]
    beat_duration = round(beat["end"] - beat["start"], 2)

    # 台词单词转为相对时间（相对于 beat 开始）
    relative_words = [
        {
            "text": w["text"],
            "start": round(w["start"] - beat["start"], 3),
            "end": round(w["end"] - beat["start"], 3),
        }
        for w in beat_words
    ]

    return f"""请生成 Beat {beat["num"]} ({beat["label"]}) 的 Hyperframes sub-composition HTML。

【基本参数】
- composition id: {beat["id"]}
- 时间范围: {beat["start"]}s - {beat["end"]}s（时长 {beat_duration}s）
- 是否最后 beat（允许淡出）: {str(is_final).lower()}

【DESIGN.md】
{design_content}

【本 Beat 分镜描述（来自 STORYBOARD.md）】
{storyboard_beat_section}

【本 Beat 台词（已转为相对时间，0 = beat 开始）】
{json.dumps(relative_words, ensure_ascii=False, indent=2)}

【代码关键要求】
const duration = {beat_duration};  // 用于所有 repeat 计算

// ★ repeat 计算固定用法（以 4s 周期为例）：
// repeat: Math.floor(duration / 4) - 1

// ★ 每个 clip 元素示例：
// <div id="{beat["id"]}-title" class="clip" data-start="0.1" data-duration="{beat_duration}"
//      data-track-index="1" style="...">文字</div>

// ★ timeline 初始化固定写法：
// window.__timelines = window.__timelines || {{}};
// const tl = gsap.timeline({{ paused: true }});
// window.__timelines["{beat["id"]}"] = tl;

请生成完整 HTML（从 <template id="{beat["id"]}-template"> 到 </template>）。"""


In [18]:
def _extract_beat_section(storyboard_content: str, beat_num: int) -> str:
    lines = storyboard_content.split("\n")
    in_section = False
    section_lines = []
    beat_header = re.compile(rf"###\s+Beat\s+{beat_num}\b", re.IGNORECASE)
    next_beat = re.compile(r"###\s+Beat\s+\d+", re.IGNORECASE)

    for line in lines:
        if beat_header.search(line):
            in_section = True
        elif in_section and next_beat.search(line) and not beat_header.search(line):
            break
        if in_section:
            section_lines.append(line)

    return "\n".join(section_lines) if section_lines else f"Beat {beat_num} 无详细描述"


In [19]:
def generate_composition_html(
    beat: dict,
    design_content: str,
    storyboard_content: str,
    transcript_words: list,
    is_final: bool,
) -> str:
    beat_section = _extract_beat_section(storyboard_content, beat["num"])
    prompt = _build_composition_prompt(
        beat, design_content, beat_section, transcript_words, is_final
    )
    system = COMPOSITION_HTML_SYSTEM_PROMPT.replace("{comp_id}", beat["id"])
    raw_html = _call_text_only(system, prompt, max_tokens=6000, temperature=0.2)

    # 去除 LLM 可能输出的 markdown 代码块包裹
    raw_html = re.sub(r"^```html?\s*", "", raw_html.strip())
    raw_html = re.sub(r"\s*```$", "", raw_html.strip())

    # 代码层面 post-process 修复
    beat_duration = round(beat["end"] - beat["start"], 2)
    fixed_html = _postprocess_composition_html(raw_html, beat["id"], beat_duration)
    return fixed_html


In [20]:
def generate_index_html(beats: list, design_content: str, total_duration: float) -> str:
    """
    根时间线 index.html。
    - 为每个 beat 生成 sub‑composition 容器
    - 扫描 audio_file/ 目录，将所有音频文件添加为全局音轨
    - 处理 track‑index 冲突，确保所有索引唯一
    """
    tokens = _extract_design_tokens(design_content)

    # ── 1. 生成 beat clips ──────────────────────────────────────
    clip_lines = []
    for i, beat in enumerate(beats):
        beat_duration = round(beat["end"] - beat["start"], 2)
        # beat clips 的 track-index 从 1 开始
        clip_lines.append(
            f"    <div\n"
            f'      id="beat-clip-{i + 1}"\n'
            f'      data-composition-id="{beat["id"]}"\n'
            f'      data-composition-src="compositions/{beat["filename"]}"\n'
            f'      data-start="{beat["start"]}"\n'
            f'      data-duration="{beat_duration}"\n'
            f'      data-width="1080"\n'
            f'      data-height="1920"\n'
            f'      data-track-index="{i + 1}"\n'
            f"    ></div>"
        )
    clips_html = "\n".join(clip_lines)

    # ── 2. 扫描 audio_file/ 目录，生成所有音频标签 ────────────────
    audio_dir = PROJECT_DIR / "audio_file"  # 与 AUDIO_DIR 常量保持一致
    audio_tags = []
    if audio_dir.exists():
        # 按文件名排序，保证顺序稳定
        audio_files = sorted(audio_dir.glob("*.*"))
        for idx, audio_path in enumerate(audio_files):
            # 只保留常见音频格式
            if audio_path.suffix.lower() not in (".mp3", ".wav", ".ogg"):
                continue
            # track-index 从 len(beats)+1 开始，避免与 beat clips (1~len(beats)) 冲突
            track_idx = len(beats) + 1 + idx
            # 简单的音量推断：文件名含 bgm/music 给 0.4，其余（如旁白）给 1.0
            vol = (
                "0.4"
                if (
                    "bgm" in audio_path.stem.lower()
                    or "music" in audio_path.stem.lower()
                )
                else "1.0"
            )
            # 相对于项目根目录的路径（index.html 所在目录）
            rel_path = audio_path.relative_to(PROJECT_DIR).as_posix()
            audio_tags.append(
                f"    <audio\n"
                f'      id="el-audio-{idx}"\n'
                f'      data-start="0"\n'
                f'      data-duration="{total_duration:.2f}"\n'
                f'      data-track-index="{track_idx}"\n'
                f'      data-volume="{vol}"\n'
                f'      src="{rel_path}"\n'
                f"    ></audio>"
            )
    bgm_html = "\n\n".join(audio_tags) if audio_tags else ""

    # ── 3. 组装完整 HTML ─────────────────────────────────────────
    return f"""<!DOCTYPE html>
<html lang="zh-CN">
<head>
  <meta charset="UTF-8" />
  <title>爆款结构迁移 — 合成视频</title>
  <style>
    * {{ margin: 0; padding: 0; box-sizing: border-box; }}
    body {{
      background: {tokens["primary"]};
      width: 1080px;
      height: 1920px;
      overflow: hidden;
    }}
  </style>
</head>
<body>
  <!-- ★ root composition 必须有 data-start="0" -->
  <div
    id="root-composition"
    data-composition-id="root"
    data-start="0"
    data-duration="{total_duration:.2f}"
    data-width="1080"
    data-height="1920"
  >
{clips_html}{bgm_html}

    <script src="https://cdn.jsdelivr.net/npm/gsap@3.14.2/dist/gsap.min.js"></script>
    <script>
      // 根时间线：各 beat sub-composition 各自管理自己的 timeline
      window.__timelines = window.__timelines || {{}};
      const tl = gsap.timeline({{ paused: true }});
      window.__timelines["root"] = tl;
    </script>
  </div>
</body>
</html>
"""


In [ ]:
def main():
    if not VIDEO_PATH.exists():
        print(f"❌ 未找到视频文件: {VIDEO_PATH}")
        sys.exit(1)

    print(f"\n{'=' * 62}")
    print("  爆款视频结构迁移引擎  v3.0")
    print(f"  视频 : {VIDEO_FILENAME}")
    print(f"  输出 : {PROJECT_DIR}")
    print(f"{'=' * 62}\n")

    # ── 阶段一：特征提取 ──────────────────────────────────────────
    print("【阶段一】提取音视频特征...")
    segments, total_text, whisper_words = extract_audio_and_transcript(
        str(VIDEO_PATH), str(OUTPUT_AUDIO_PATH)
    )
    cuts, fps, video_duration = analyze_video_rhythm(str(VIDEO_PATH))
    print(
        f"✅ 时长: {video_duration:.1f}s | 切镜: {len(cuts)}个 | "
        f"段落: {len(segments)}段 | 单词: {len(whisper_words)}个"
    )

    # ── 编码视频（Steps A/B/C 使用多模态）────────────────────────
    video_b64 = encode_video_base64(str(VIDEO_PATH))

    # ── Step A: DESIGN.md ─────────────────────────────────────────
    print("\n【Step A】生成 DESIGN.md ...")
    design_prompt = build_design_prompt(segments, total_text, cuts, video_duration)
    design_content = _call_multimodal(DESIGN_SYSTEM_PROMPT, design_prompt, video_b64)
    DESIGN_MD_PATH.write_text(design_content, encoding="utf-8")
    print(f"✅ DESIGN.md → {DESIGN_MD_PATH}")

    # ── Step B: SCRIPT.md ─────────────────────────────────────────
    print("\n【Step B】生成 SCRIPT.md ...")
    script_prompt = build_script_prompt(
        segments, total_text, cuts, video_duration, design_content
    )
    script_content = _call_multimodal(SCRIPT_SYSTEM_PROMPT, script_prompt, video_b64)
    SCRIPT_MD_PATH.write_text(script_content, encoding="utf-8")
    print(f"✅ SCRIPT.md → {SCRIPT_MD_PATH}")

    # ── Step C: STORYBOARD.md ────────────────────────────────────
    print("\n【Step C】生成 STORYBOARD.md ...")
    sb_prompt = build_storyboard_prompt(
        segments, cuts, video_duration, design_content, script_content
    )
    storyboard_content = _call_multimodal(
        STORYBOARD_SYSTEM_PROMPT, sb_prompt, video_b64
    )
    STORYBOARD_MD_PATH.write_text(storyboard_content, encoding="utf-8")
    print(f"✅ STORYBOARD.md → {STORYBOARD_MD_PATH}")

    del video_b64  # 释放内存，后续步骤不需要多模态

    # ── Step D: narration.txt + transcript.json ───────────────────
    print("\n【Step D】生成 narration.txt + transcript.json ...")
    narration_clean = generate_narration_txt(total_text)
    NARRATION_TXT_PATH.write_text(narration_clean, encoding="utf-8")
    print(f"✅ narration.txt → {NARRATION_TXT_PATH}")

    transcript_data = build_transcript_json(whisper_words)
    TRANSCRIPT_JSON_PATH.write_text(
        json.dumps(transcript_data, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(
        f"✅ transcript.json → {TRANSCRIPT_JSON_PATH}  ({len(transcript_data)} 个单词)"
    )

    # ── Step E: compositions/*.html + index.html ──────────────────
    print("\n【Step E】解析 Beat 结构并生成 compositions/ ...")
    beats = _parse_beats_from_storyboard(storyboard_content)

    if not beats:
        print("⚠️  STORYBOARD.md 中未解析到 Beat，按切镜点自动分割...")
        boundary_times = sorted(set([0.0] + cuts[:8] + [video_duration]))
        label_map = {0: "hook", len(boundary_times) - 2: "cta"}
        for i in range(len(boundary_times) - 1):
            label = label_map.get(i, f"story-{i}")
            beats.append(
                {
                    "num": i + 1,
                    "id": f"beat-{i + 1}-{label}",
                    "label": label.capitalize(),
                    "start": boundary_times[i],
                    "end": boundary_times[i + 1],
                    "filename": f"beat-{i + 1}-{label}.html",
                    "narration": "",
                }
            )

    print(f"   检测到 {len(beats)} 个 Beat")

    for idx, beat in enumerate(beats):
        is_final = idx == len(beats) - 1
        beat_file = COMPOSITIONS_DIR / beat["filename"]
        print(f"   → 生成 {beat['filename']} (Beat {beat['num']}: {beat['label']}) ...")
        html_content = generate_composition_html(
            beat, design_content, storyboard_content, whisper_words, is_final
        )
        beat_file.write_text(html_content, encoding="utf-8")
        print(f"     ✅ {beat_file}")

    print("\n   → 生成 index.html ...")
    index_content = generate_index_html(beats, design_content, video_duration)
    INDEX_HTML_PATH.write_text(index_content, encoding="utf-8")
    print(f"✅ index.html → {INDEX_HTML_PATH}")

    # ── 完成摘要 ─────────────────────────────────────────────────
    print(f"\n{'=' * 62}")
    print("  🎉 全部完成！")
    print(f"  DESIGN.md        → {DESIGN_MD_PATH}")
    print(f"  SCRIPT.md        → {SCRIPT_MD_PATH}")
    print(f"  STORYBOARD.md    → {STORYBOARD_MD_PATH}")
    print(f"  narration.txt    → {NARRATION_TXT_PATH}")
    print(f"  transcript.json  → {TRANSCRIPT_JSON_PATH}")
    for b in beats:
        print(f"  compositions/{b['filename']}")
    print(f"  index.html       → {INDEX_HTML_PATH}")
    print(f"{'=' * 62}")
    print()
    print("下一步（在 my-video/ 目录下执行）：")
    print("  npx hyperframes lint        # 应零 error")
    print("  npx hyperframes validate    # 运行时检查")
    print("  npx hyperframes preview     # 本地预览")
    print("  # 如需 TTS 配音：")
    print("  npx hyperframes tts SCRIPT.md --voice af_nova --output narration.wav")
    print("  npx hyperframes render --output output.mp4")
    print()


if __name__ == "__main__":
    import traceback

    try:
        main()
    except Exception as e:
        traceback.print_exc()
        print(f"\n❌ 流水线失败: {e}")
        sys.exit(1)



  爆款视频结构迁移引擎  v3.0
  视频 : 抖音2026529-207530.mp4
  输出 : s:\Project\video_structure_transform\backend\my-video

【阶段一】提取音视频特征...
📢 正在提取音频...
🗣️  正在 ASR 转录...


s:\Project\video_structure_transform\backend\.venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


🎬 正在分析视频节奏...
✅ 时长: 20.1s | 切镜: 63个 | 段落: 6段 | 单词: 6个
💾 正在将视频编码为 base64...

【Step A】生成 DESIGN.md ...
✅ DESIGN.md → s:\Project\video_structure_transform\backend\my-video\DESIGN.md

【Step B】生成 SCRIPT.md ...
✅ SCRIPT.md → s:\Project\video_structure_transform\backend\my-video\SCRIPT.md

【Step C】生成 STORYBOARD.md ...
✅ STORYBOARD.md → s:\Project\video_structure_transform\backend\my-video\STORYBOARD.md

【Step D】生成 narration.txt + transcript.json ...
✅ narration.txt → s:\Project\video_structure_transform\backend\my-video\narration.txt
✅ transcript.json → s:\Project\video_structure_transform\backend\my-video\transcript.json  (6 个单词)

【Step E】解析 Beat 结构并生成 compositions/ ...
   检测到 5 个 Beat
   → 生成 beat-1-hook.html (Beat 1: Hook) ...
     ✅ s:\Project\video_structure_transform\backend\my-video\compositions\beat-1-hook.html
   → 生成 beat-2-story.html (Beat 2: Story) ...
     ✅ s:\Project\video_structure_transform\backend\my-video\compositions\beat-2-story.html
   → 生成 beat-3-climax.html (Beat 3: Cl

Traceback (most recent call last):
  File "C:\Users\Ryan Wang\AppData\Local\Temp\ipykernel_40404\1211489239.py", line 140, in <module>
    main()
  File "C:\Users\Ryan Wang\AppData\Local\Temp\ipykernel_40404\1211489239.py", line 104, in main
    index_content = generate_index_html(
                    ^^^^^^^^^^^^^^^^^^^^
TypeError: generate_index_html() got an unexpected keyword argument 'audio_source_path'


SystemExit: 1

s:\Project\video_structure_transform\backend\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
